# LLM最关键的设计（Attention Is All You Need）

来自视频 [LLM最关键的设计（Attention Is All You Need）](https://www.bilibili.com/video/BV1DW421R7rz) ，从这一章开始就要跟着老师开始从零实现 GPT2 了。

另外，貌似 token 的解释可以对应到中文的 “词元” 。

在之前的课程学习中，我们学习 RNN 进行文本翻译任务的时候，是尝试让编码器将一句话编码，并取出最后一个隐藏状态，这个隐藏状态是包含了前文所有上下文信息的，将这个隐藏状态传递给解码器就可以进行翻译的输出解码。但是问题在于，隐藏状态的特征是有限的，假设文本很长，隐藏状态就难以捕获所有信息，而且里隐藏状态越远的文本所获得的信息就越少，这样在长文本翻译任务的时候就不准确了。所以就有人提出，是否可以将所有的隐藏状态一同进行计算，这样就可以解决这个问题。在解码其中，对于当前的输入分别计算每一个隐藏状态的权重，在权重的基础上加和所有的隐藏状态，得到所谓的“背景向量”给到解码器（作为附加的输入），这就是注意力机制。

![](./images/注意力机制.png)

![](./images/注意力机制的发展史.png)

$I_i$ 是解码器当前输入词元的状态，通过 $I_i$ 与各个编码器输出的隐藏状态的计算目的是（计算出与输出最有可能性的隐藏状态并）加权。

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [34]:
sequence_len = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [35]:
# softmax 特性
a = torch.tensor((1, 2, float('-inf'))).float()
print(F.softmax(a, dim=-1))

tensor([0.2689, 0.7311, 0.0000])


In [36]:
# K: (B, T, H)  B -- 批量大小  T -- 文本长度  H -- 特征个数
# Q: (B, T, H)
# 对齐分数： K @ Q.transpose(-2, -1) （内积为矩阵乘法）  transpose 交换维度
# K @ Q.transpose(-2, -1)  (B, T, T)
scores = torch.randn(1, 4, 4)
scores

tensor([[[ 0.6087,  0.8096,  0.6145, -0.2689],
         [-1.6752,  1.1493, -0.4753, -1.9654],
         [ 0.6156,  0.1876,  0.6369, -1.0195],
         [-0.5846, -0.5445, -0.4847, -0.6873]]])

In [37]:
# 定义下三角矩阵
tril = torch.tril(torch.ones(4, 4))
print(tril)
s = scores.masked_fill(tril == 0, float('-inf'))
s

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


tensor([[[ 0.6087,    -inf,    -inf,    -inf],
         [-1.6752,  1.1493,    -inf,    -inf],
         [ 0.6156,  0.1876,  0.6369,    -inf],
         [-0.5846, -0.5445, -0.4847, -0.6873]]])

In [38]:
# 定义权重分布
print(F.softmax(s, dim=-1))

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.0560, 0.9440, 0.0000, 0.0000],
         [0.3741, 0.2438, 0.3821, 0.0000],
         [0.2470, 0.2571, 0.2730, 0.2229]]])


In [39]:
# softmax 对于标准差的敏感性
x = torch.rand(1, 8)
print(x.std(), F.softmax(x, dim=-1))
print(F.softmax(x * 1000, dim=-1))  # 对方差敏感，标准差必须趋于1

tensor(0.2665) tensor([[0.1693, 0.1284, 0.1339, 0.1558, 0.0904, 0.0793, 0.1036, 0.1391]])
tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 1.0554e-36, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00]])


In [40]:
# 对齐分数的标准差变化
B, T, H = 32, 100, 10
K = torch.randn(B, T, H)
Q = torch.randn(B, T, H)
scores = K @ Q.transpose(-2, -1) / H ** 0.5  # 标准差趋于 1，归一化处理
print(scores.std())

tensor(0.9888)


![](./images/注意力机制Transformer.png)
![](./images/注意力机制代码.png)

In [41]:
def attention(query, key, value, dropout, mask=None):  # make=None 双向注意力，下三角为单向自注意力机制
    # query, key, value: (B, T, H)
    # mask: (T, T)
    # output: (B, T, H)
    B, T, H = query.shape
    scores = query @ key.transpose(-2, -1) / H ** 0.5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    w_att = F.softmax(scores, dim=-1)  # (B, T, T)
    w_att = dropout(w_att)

    # 计算背景向量
    out = w_att @ value  # (B, T, H)
    return out

这行代码是 **PyTorch** 中常见的用法，特别是在实现 **Transformer 的因果注意力（causal self-attention）** 时会用到。我们来逐部分解析它的含义：

```python
self.register_buffer('tril', torch.tril(torch.ones(sequence_len, sequence_len)))
```

---

### 🔍 分解解释

#### 1. `torch.ones(sequence_len, sequence_len)`
- 创建一个形状为 `(sequence_len, sequence_len)` 的全 1 矩阵（即单位矩阵的“1版”）。
- 例如，如果 `sequence_len = 4`，结果是：
  $$
  \begin{bmatrix}
  1 & 1 & 1 & 1 \\
  1 & 1 & 1 & 1 \\
  1 & 1 & 1 & 1 \\
  1 & 1 & 1 & 1 \\
  \end{bmatrix}
  $$

#### 2. `torch.tril(...)`
- `tril` 是 **"lower triangular"** 的缩写，表示“下三角矩阵”。
- 它会把矩阵中**主对角线及以上保留，主对角线以下设为 0**。
- 对上面的矩阵应用 `torch.tril` 后变成：
  $$
  \begin{bmatrix}
  1 & 0 & 0 & 0 \\
  1 & 1 & 0 & 0 \\
  1 & 1 & 1 & 0 \\
  1 & 1 & 1 & 1 \\
  \end{bmatrix}
  $$
  > ✅ 注意：默认 `upper_diagonal=0`，即只保留主对角线和下方元素。

  （可选参数：`torch.tril(matrix, diagonal=k)`，k=0 是主对角线，k=1 包含上一行等）

#### 3. `self.register_buffer(...)`
这是 PyTorch 的一个方法，用于：
- 将某个张量注册为模型的一个 **“buffer”（缓冲区）**
- 这个张量会：
  - 被保存在模型的状态字典中（`state_dict`）
  - 随模型一起被 `to(device)` 移动（比如从 CPU 到 GPU）
  - **但不会被当作可训练参数（不会参与梯度更新）**

> ✅ 用途：适合存放那些**固定或预定义的辅助张量**，比如位置编码、掩码（mask）等。

---

### 🎯 整体作用：创建一个**因果注意力掩码（causal mask）**

这个 `tril` 缓冲区通常用于 **防止模型在预测时看到未来的信息**。

#### ✅ 应用场景：语言模型自回归生成

比如你在生成一句话：“我 爱 机 器 学 习”，模型应该：

- 在预测“爱”时，只能看“我”
- 在预测“机”时，只能看“我 爱”
- ...
- 不能偷看未来的词！

所以，在计算注意力分数时，要用这个 `tril` 掩码来屏蔽未来位置。

#### ✅ 代码示例（注意力中的使用）：

```python
# 假设 attention_scores 形状为 (batch, heads, seq_len, seq_len)
attention_scores = ...  # Q @ K.T

# 应用掩码：把未来位置的分数设为非常小的数（如 -inf）
mask = self.tril[:seq_len, :seq_len]  # 取所需大小
attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

# 然后 softmax
attention_weights = F.softmax(attention_scores, dim=-1)
```

这样，softmax 之后，被掩掉的位置（未来）权重接近 0。

---

### 📌 总结：这行代码的意思是

> 将一个 `sequence_len × sequence_len` 的**下三角矩阵**（主对角线及以下为1，其余为0）注册为模型的一个**不可训练的缓冲区**，命名为 `'tril'`，用于后续实现**因果注意力机制**，防止模型关注未来时间步。

---

### 💡 补充建议

- 实际中 `sequence_len` 不一定是固定的，可以设大一点（如 1024），然后运行时动态切片。
- 更灵活的做法是延迟创建掩码，直到前向传播时根据实际序列长度生成。

```python
# 更动态的方式（推荐）
def forward(self, x):
    T = x.size(1)
    mask = torch.tril(torch.ones(T, T, device=x.device))
    ...
```

但 `register_buffer` 的方式适合固定长度或训练期间不变的情况。



In [42]:
class MaskedAttention(nn.Module):
    # 单向自注意力

    def __init__(self, emb_size, head_size):
        """
        :param emb_size: 输入向量长度 -- C
        :param head_size: 背景向量长度 -- H
        """
        super().__init__()
        # 不需要截距项之一是因为大语言模型的训练中都有残差项来加速训练
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(sequence_len, sequence_len)))
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x: (B, T, C)
        # out: (B, T, H)
        B, T, C = x.shape
        k = self.key(x)  # (B, T, H)
        q = self.query(x)   # (B, T, H)
        v = self.value(x)  # (B, T, H)

        mask = self.tril[:T, :T]  # 截取一块和文本长度一样大小的掩码，如果 T 超过 sequence_len 就会报错，这是注意力机制的一个缺陷
        out = attention(q, k, v, self.dp, mask)
        return out

In [43]:
m = MaskedAttention(3, 4)
x = torch.randn(5, 10, 3)
m(x).shape

torch.Size([5, 10, 4])

In [66]:
print(nn.Dropout(0.4)(F.softmax(torch.ones(5, 5), dim=-1)))

tensor([[0.0000, 0.3333, 0.3333, 0.0000, 0.3333],
        [0.3333, 0.0000, 0.0000, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.3333, 0.0000],
        [0.3333, 0.0000, 0.3333, 0.3333, 0.3333]])
